# DataCoolie Databricks Secret Scope Validation

Small notebook to validate Databricks secret-scope access separately from file/delta orchestration.

This sample does not print secret values. It only verifies retrieval and resolution behavior.

In [ ]:
%pip install "datacoolie" --upgrade
# Optional local wheel install:
# %pip install /Volumes/workspace/default/datacoolie_sim/libraries/datacoolie-0.1.0-py3-none-any.whl --force-reinstall

In [ ]:
dbutils.library.restartPython()  # type: ignore[name-defined]

In [ ]:
# Create or update a Databricks secret using Databricks SDK for Python.
# The next cell reuses SECRET_SCOPE and SECRET_KEY from this cell.

import os

try:
    from databricks.sdk import WorkspaceClient  # type: ignore[import-not-found]
except ImportError as exc:
    raise ImportError(
        "databricks-sdk is not installed. Run: %pip install databricks-sdk"
    ) from exc

SECRET_SCOPE = "datacoolie-scope"
SECRET_KEY = "sample-db-password"
SECRET_VALUE = "DataCoolie@1"
if not SECRET_VALUE:
    raise ValueError(
        "No secret value provided. Set widget 'secret_value' or env var DATACOOLIE_SECRET_VALUE."
    )

w = WorkspaceClient()

_scope_exists = any(s.name == SECRET_SCOPE for s in w.secrets.list_scopes())
if not _scope_exists:
    w.secrets.create_scope(scope=SECRET_SCOPE)
    print(f"Created scope: {SECRET_SCOPE}")
else:
    print(f"Scope already exists: {SECRET_SCOPE}")

# put_secret upserts: creates key if missing, overwrites value if key exists.
w.secrets.put_secret(scope=SECRET_SCOPE, key=SECRET_KEY, string_value=SECRET_VALUE)

print(
    "Secret upsert completed.",
    {
        "scope": SECRET_SCOPE,
        "key": SECRET_KEY,
        "value_length": len(SECRET_VALUE),
    },
)

In [ ]:
from datacoolie.core.models import Connection
from datacoolie.core.secret_provider import resolve_secrets
from datacoolie.platforms.databricks_platform import DatabricksPlatform

# Reuse values from the SDK cell if present; otherwise fall back to defaults.
SECRET_SCOPE = globals().get("SECRET_SCOPE", "datacoolie-scope")
SECRET_KEY = globals().get("SECRET_KEY", "sample-db-password")

platform = DatabricksPlatform()

# 1) Direct provider validation (dbutils.secrets via DatabricksPlatform).
raw_value = platform.get_secret(key=SECRET_KEY, source=SECRET_SCOPE)
if not raw_value:
    raise RuntimeError("Secret value is empty. Check scope/key and permissions.")

# 2) DataCoolie-style connection resolution with secrets_ref.
conn = Connection(
    name="databricks_secret_probe",
    connection_type="database",
    format="sql",
    configure={"database_type": "postgresql", "password": SECRET_KEY},
    secrets_ref={SECRET_SCOPE: ["password"]},
)
resolve_secrets(conn, platform)

resolved_value = conn.configure.get("password", "")
if not resolved_value:
    raise RuntimeError("Secret resolution failed for configure.password.")

print("Secret scope validation passed.")
print({
    "scope": SECRET_SCOPE,
    "key": SECRET_KEY,
    "resolved_field": "password",
    "length": len(resolved_value),
    "matches_direct_fetch": resolved_value == raw_value,
})